# **SDP-Symresack: Symmetry Handling in Semidefinite Programming**

This notebook extends the concept of **symresacks**<sup>[[3]](#ref3)</sup>, originally developed for lexicographically maximal orbit representatives in binary integer programs, to semidefinite programming. Using the maximum cut SDP relaxation on the complete graph $K_3$, we compare three symmetry handling approaches analytically, without external solvers, so that every step can be verified by hand. The core contribution is the introduction of **FD-inequalities**, which are lexicographic ordering constraints on matrix entries that select a canonical matrix per symmetry orbit while preserving the SDP optimal value, in contrast to group averaging methods that may degrade the bound.

---

## *MaxCut SDP Relaxation on $K_3$*

Given an undirected graph $G = (V, E)$ with edge weights $w_{ij} > 0$, the maximum cut problem seeks a partition $V = S \cup \bar{S}$ that maximizes the total weight of edges crossing the cut.<sup>[[1]](#ref1)</sup> The Goemans-Williamson semidefinite relaxation replaces each vertex $i$ with a unit vector $\mathbf{v}_i \in \mathbb{R}^n$ and encodes the partition via inner products $X_{ij} = \mathbf{v}_i \cdot \mathbf{v}_j$:

\begin{equation*}
\begin{aligned}
\max \quad & \frac{1}{2} \sum_{i < j} w_{ij} (1 - X_{ij}) \\
\text{s.t.} \quad & X_{ii} = 1, \quad \forall i \in V \\
 & X \succeq 0
\end{aligned}
\end{equation*}

The matrix $X \in \mathbb{S}^n_+$ is positive semidefinite (PSD) with unit diagonal. For any feasible $X$, the objective gives an upper bound on the true max-cut value. When the vectors $\mathbf{v}_i$ lie in $\mathbb{R}^1$, the entries take values in $\{-1, 1\}$ and the bound is exact.

### **$K_3$ with Unit Weights**

Consider the complete graph $K_3$ with vertices $V = \{1, 2, 3\}$ and unit weights $w_{ij} = 1$ for all $i < j$. The SDP matrix takes the form:

\begin{equation*}
X = \begin{bmatrix} 1 & a & b \\ a & 1 & c \\ b & c & 1 \end{bmatrix}
\end{equation*}

Positive semidefiniteness is equivalent to all principal minors being non-negative, which for a $3 \times 3$ matrix reduces to:

\begin{equation*}
-1 \leq a, b, c \leq 1, \qquad
\det X = 1 + 2abc - a^2 - b^2 - c^2 \geq 0
\end{equation*}

The objective becomes $(3 - a - b - c)/2$. The true maximum cut of $K_3$ has value $2$, achieved by the $2$-$1$ partition $S = \{1, 2\}$, $\bar{S} = \{3\}$, which corresponds to:

\begin{equation*}
X_{\text{opt}} = \begin{bmatrix}
1 & 1 & -1 \\
1 & 1 & -1 \\
-1 & -1 & 1
\end{bmatrix}, \qquad (a, b, c) = (1, -1, -1)
\end{equation*}

**Verification.** $\det X_{\text{opt}} = 1 + 2(1)(-1)(-1) - 1 - 1 - 1 = 0$ ✓. The objective is $(3 - 1 + 1 + 1)/2 = 2$. Since the SDP relaxation cannot exceed the true max-cut value, $2$ is optimal.

### **$S_3$ Symmetry by Conjugation**

The complete graph $K_3$ is vertex-transitive: the symmetric group $S_3$ acts on the vertices $\{1, 2, 3\}$.<sup>[[2]](#ref2)</sup> For a permutation $\pi \in S_3$ with permutation matrix $P_\pi$, the action on the SDP matrix $X$ is by conjugation:

\begin{equation*}
X \; \mapsto \; P_\pi X P_\pi^{\mathsf{T}}
\end{equation*}

This permutes both rows and columns simultaneously, shuffling the three off-diagonal entries $\{a, b, c\} = \{X_{12}, X_{13}, X_{23}\}$ among themselves. Since $X_{ii} = 1$, the diagonal is invariant. The PSD property is preserved under permutation similarity, and the objective $(3 - a - b - c)/2$ depends only on the multiset $\{a, b, c\}$, so all $|S_3| = 6$ conjugates of any feasible $X$ are equally feasible and attain the same objective value.

For $X_{\text{opt}}$, the six orbit elements correspond to placing the single $+1$ in any of the three off-diagonal positions, with the remaining transposition generating two matrices each.

### *GP1 Averaging: Fixed-Point Restriction*

Gatermann and Parrilo<sup>[[4]](#ref4)</sup> propose symmetry reduction for SDP by restricting the feasible region to the **fixed-point subspace** of the group action, i.e., matrices invariant under all $\pi \in S_3$:

\begin{equation*}
X = P_\pi X P_\pi^{\mathsf{T}}, \quad \forall \pi \in S_3
\end{equation*}

Since $S_3$ acts transitively on the three off-diagonal entries, this forces all of them equal:

\begin{equation*}
a = b = c := t, \qquad
X_{\text{GP1}} = \begin{bmatrix} 1 & t & t \\ t & 1 & t \\ t & t & 1 \end{bmatrix}
\end{equation*}

The PSD condition reduces to $\det X_{\text{GP1}} = (1 - t)^2(1 + 2t) \geq 0$, which implies $t \geq -\frac{1}{2}$. The objective becomes $\frac{3}{2}(1 - t)$, decreasing in $t$, so the best possible value is attained at the PSD boundary $t = -\frac{1}{2}$:

\begin{equation*}
\frac{3}{2}\!\left(1 + \frac{1}{2}\right) = \frac{9}{4} = 2.25
\end{equation*}

The GP1 bound is $2.25$, strictly weaker than the true optimal value $2$. The restriction eliminates $X_{\text{opt}}$ because its off-diagonal entries $(1, -1, -1)$ are not all equal, forcing the solver to accept a worse solution.

### *FD-Inequalities: The SDP-Symresack*

Rather than averaging symmetric variables together, we can order them lexicographically, following the same principle that symresacks<sup>[[3]](#ref3)</sup> apply to binary integer programs. For $S_3$ acting on $X \in \mathbb{S}^3_+$, vectorize by reading the upper triangle row by row:

\begin{equation*}
\operatorname{vec}(X) = (X_{11}, X_{12}, X_{13}, X_{22}, X_{23}, X_{33})
\end{equation*}

The lexicographically maximal matrix in each orbit is the one where the first off-diagonal entry $(X_{12} = a)$ dominates the others. The corresponding **FD-inequality** (named after the first-different entry in the vectorization) is:

\begin{equation*}
a \geq b
\end{equation*}

This single linear constraint suffices because $S_3$ can map any off-diagonal entry to the $(1,2)$ position. With the FD-inequality $a \geq b$, the SDP-symresack becomes:

\begin{equation*}
\begin{aligned}
\max \quad & \frac{3 - a - b - c}{2} \\
\text{s.t.} \quad & X \succeq 0, \quad X_{ii} = 1 \\
& a \geq b
\end{aligned}
\end{equation*}

**Verification.** The optimal matrix $X_{\text{opt}}$ with $(a, b, c) = (1, -1, -1)$ satisfies $1 \geq -1$ ✓ and remains feasible. The objective value is $2$, unchanged from the unconstrained SDP.

| Approach | Bound | Preserves optimality? |
|----------|-------|----------------------|
| No SB    | 2     | baseline              |
| GP1 avg  | 2.25  | No, degrades          |
| FD-ineq  | 2     | Yes, preserves        |

---

The $K_3$ example illustrates the core trade-off. Group averaging collapses symmetric variables to reduce dimension, but too aggressively: it forces the solution to be fully $G$-invariant, which can exclude the true optimum when that optimum is not symmetric. FD-inequalities select one canonical matrix per orbit via linear ordering constraints while preserving the optimal value.

Extending this beyond $K_3$ requires a chain of $O(n^2)$ FD-inequalities for full $S_n$ symmetry ($X_{12} \geq X_{13} \geq \cdots \geq X_{1n}$, then $X_{23} \geq X_{24} \geq \cdots$, etc.). For a single cyclic generator, the inequalities reduce to $O(n)$ per orbit. For signed permutation groups $B_n$, absolute-value FD-inequalities ($|X_{12}| \geq |X_{13}| \geq \cdots$) are needed.

The analytical approach taken here, working with a $3 \times 3$ matrix whose PSD conditions reduce to a single cubic inequality, makes every claim directly verifiable without solvers or external software.

### **References**

<a id="ref1"></a>
<sup>[[1]](#ref1)</sup> Goemans, M. X., & Williamson, D. P. (1995). Improved approximation algorithms for maximum cut and satisfiability problems using semidefinite programming. *Journal of the ACM*, 42(6), 1115–1145.

<a id="ref2"></a>
<sup>[[2]](#ref2)</sup> Hojny, C., & Pfetsch, M. E. (2019). Polytopes associated with symmetry handling. *Mathematical Programming*, 175, 197–240.

<a id="ref3"></a>
<sup>[[3]](#ref3)</sup> Hojny, C. (2020). Packing, partitioning, and covering symresacks. *Discrete Applied Mathematics*, 283, 689–717.

<a id="ref4"></a>
<sup>[[4]](#ref4)</sup> Gatermann, K., & Parrilo, P. A. (2004). Symmetry groups, semidefinite programs, and sums of squares. *Journal of Pure and Applied Algebra*, 192(1–3), 95–128.

<a id="ref5"></a>
<sup>[[5]](#ref5)</sup> Margot, F. (2010). Symmetry in integer linear programming. In *50 Years of Integer Programming 1958–2008* (pp. 647–686). Springer.

<a id="ref6"></a>
<sup>[[6]](#ref6)</sup> Kobert, P., & Scheiderer, C. (2020). Spectrahedral representation of polar orbitopes. arXiv:2010.02045.